# Fine-Tuning XLM-RoBERTa untuk Emotion Detection (Bahasa Indonesia)

Notebook ini berisi langkah-langkah untuk melakukan fine-tuning model **XLM-RoBERTa** (`xlm-roberta-base`) pada dataset **IndoNLU EmoT** untuk klasifikasi 5 emosi: *sadness, anger, love, fear, happy*. XLM-RoBERTa merupakan pre-trained model multilingual berbasis RoBERTa yang dilatih pada 100+ bahasa termasuk Bahasa Indonesia, sehingga menjadi pembanding yang kuat untuk model monolingual.

In [ ]:
# 1. Install dependensi jika dijalankan di Google Colab
# Gunakan requirements.txt untuk menjamin kompatibilitas versi pustaka
# !pip install -q -r ../requirements.txt

In [ ]:
import sys
import os
import csv
import numpy as np
import torch
from transformers import AutoTokenizer

# Menambahkan path root project agar modul src bisa di-import
sys.path.append(os.path.abspath(".."))

from src.data import load_emot_dataset, tokenize_dataset
from src.train import build_trainer
from src.evaluate import detailed_classification_report, plot_confusion_matrix, plot_training_curves

## 2. Load dan Tokenisasi Dataset

Kita memuat dataset `emot` dan melakukan pembersihan serta tokenisasi menggunakan tokenizer pendamping dari XLM-RoBERTa.

In [ ]:
model_name = "xlm-roberta-base"

# Muat dataset
dataset = load_emot_dataset()
label_names = dataset["train"].features["label"].names

# Load tokenizer pendamping model XLM-RoBERTa
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenisasi dataset
tokenized_dataset = tokenize_dataset(dataset, tokenizer, max_length=96)

print("Kolom setelah tokenisasi:", tokenized_dataset["train"].column_names)

## 3. Konfigurasi Trainer & Baseline Fine-Tuning

Kita akan melatih model menggunakan `build_trainer` dengan hyperparameter baseline:
- `learning_rate`: 2e-5
- `batch_size`: 16
- `epochs`: 5

**Catatan**: Parameter-parameter ini adalah baseline awal. Eksplorasi hyperparameter secara terstruktur dapat dilihat pada notebook `04_hyperparameter_tuning.ipynb`.

In [ ]:
output_dir = "../results/xlmr_baseline"
learning_rate = 2e-5
batch_size = 16
num_epochs = 5

trainer = build_trainer(
    model_name=model_name,
    tokenized_dataset=tokenized_dataset,
    output_dir=output_dir,
    learning_rate=learning_rate,
    batch_size=batch_size,
    num_epochs=num_epochs,
    seed=42
)

## 4. Eksekusi Training

*PENTING: Jangan jalankan training penuh di komputer lokal tanpa GPU (M1 8GB RAM) karena akan sangat lambat dan berisiko overhead. Jalankan cell ini di Google Colab GPU.*

In [ ]:
# Mulai proses training model
trainer.train()

## 5. Evaluasi dan Penyimpanan Hasil

Setelah training selesai, kita mengevaluasi model pada data test, memvisualisasikan performa, dan mencatat metriknya.

In [ ]:
# Melakukan prediksi pada test set
test_predictions = trainer.predict(tokenized_dataset["test"])
print("Metrik Prediksi Test:", test_predictions.metrics)

# Mengambil label prediksi dan label aktual
y_pred = np.argmax(test_predictions.predictions, axis=-1)
y_true = tokenized_dataset["test"]["label"]

# 1. Print classification report secara detail
report = detailed_classification_report(y_true, y_pred, label_names)
print("=== CLASSIFICATION REPORT XLM-RoBERTa ===")
print(report)

# 2. Plot dan simpan Confusion Matrix
plot_confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    label_names=label_names,
    save_path="../results/figures/xlmr_confusion_matrix.png"
)

# 3. Plot dan simpan Kurva Training (Loss & F1)
plot_training_curves(
    log_history=trainer.state.log_history,
    save_path="../results/figures/xlmr_training_curves.png"
)

In [ ]:
# 4. Mencatat metrik performa model ke file CSV gabungan
accuracy = test_predictions.metrics.get("test_accuracy", 0.0)
f1_macro = test_predictions.metrics.get("test_f1_macro", 0.0)

row_data = {
    "model": "XLM-RoBERTa-base",
    "learning_rate": learning_rate,
    "batch_size": batch_size,
    "epochs": num_epochs,
    "accuracy": round(accuracy, 4),
    "f1_macro": round(f1_macro, 4)
}

metrics_csv_path = "../results/metrics.csv"
file_exists = os.path.exists(metrics_csv_path) and os.path.getsize(metrics_csv_path) > 0

with open(metrics_csv_path, mode="a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row_data.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row_data)

print("Hasil metrik XLM-RoBERTa baseline berhasil dicatat ke results/metrics.csv")